⚠️ **Static Version Notice**

This is a static export of an interactive marimo notebook. Some features have been modified for compatibility:

- Interactive UI elements (sliders, dropdowns, text inputs) have been removed
- UI variable references have been replaced with default values
- Some cells may have been simplified or removed entirely

For the full interactive experience, please run the original marimo notebook (.py file) using:
```bash
uv run marimo edit notebook_name.py
```

---


# Module 9: Practical - Music Generation Transformer

In this practical, we'll build a transformer model for music generation. Similar to text generation,
we'll train a model to predict the next note in a sequence, enabling it to compose original melodies.

We'll use a simple piano roll representation where each note is represented by its MIDI pitch value (0-127).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

# Check which GPU is available
device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
)

print(f"Using device: {device}")


In [ ]:
import glob
import os
import pretty_midi

def load_midi_files(midi_dir=None):
    """Load MIDI files from a directory and extract note sequences"""
    all_notes = []

    if os.path.exists(midi_dir):
        midi_files = glob.glob(os.path.join(midi_dir, "*.mid")) + \
                     glob.glob(os.path.join(midi_dir, "*.midi"))

        print(f"Found {len(midi_files)} MIDI files")

        for midi_file in midi_files[:20]:  # Limit to first 20 files
            try:
                midi_data = pretty_midi.PrettyMIDI(midi_file)

                # Extract notes from all instruments
                for instrument in midi_data.instruments:
                    if not instrument.is_drum:  # Skip drum tracks
                        # Sort notes by start time
                        notes = sorted(instrument.notes, key=lambda x: x.start)

                        # Extract pitch values
                        for note in notes:
                            all_notes.append(note.pitch)

            except Exception as e:
                print(f"Error loading {midi_file}: {e}")
                continue

        if len(all_notes) > 0:
            print(f"Extracted {len(all_notes)} notes from MIDI files")
            return all_notes

    print("Loading MIDI failed")

midi_directory = "../data/maestro-v1.0.0/2015"  # Set to "./midi_files" or your MIDI folder path

music_data = load_midi_files(midi_directory)
print(f"Total musical events: {len(music_data)}")
print(f"Sample notes: {music_data[:30]}")
print(f"Note range: {min(music_data)} to {max(music_data)}")


## Vocabulary and Tokenization

For music, our vocabulary consists of:
- MIDI note values (0-127)
- Special tokens: <PAD> (padding), <START> (sequence start), <END> (sequence end)

In [ ]:
# Create vocabulary for music
# MIDI notes range from 0-127, we'll add special tokens
VOCAB_SIZE = 131  # 128 MIDI notes + <PAD>=128, <START>=129, <END>=130
PAD_TOKEN = 128
START_TOKEN = 129
END_TOKEN = 130

# Map for decoding
def decode_note(token):
    if token == PAD_TOKEN:
        return "<PAD>"
    elif token == START_TOKEN:
        return "<START>"
    elif token == END_TOKEN:
        return "<END>"
    else:
        return f"Note{token}"

# Encode the music data (already in MIDI format)
encoded_music = music_data.copy()

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Encoded sequence length: {len(encoded_music)}")


## Sequence Creation

We'll create fixed-length sequences of musical events. Each sequence will be used to predict
the next note, similar to language modeling.

In [ ]:
# Create sequences for music generation
SEQ_LEN = 32  # Sequence length for music patterns

class MusicDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
            torch.tensor(self.data[idx:idx + self.seq_len], dtype=torch.long),
            torch.tensor(self.data[idx + 1:idx + self.seq_len + 1], dtype=torch.long)
        )

music_dataset = MusicDataset(encoded_music, SEQ_LEN)
music_loader = DataLoader(music_dataset, batch_size=32, shuffle=True)

print(f"Dataset size: {len(music_dataset)} sequences")
print(f"Batch size: 32")


Let's examine a sample batch of sequences:

In [ ]:
sample_batch = next(iter(music_loader))
print(f"Input shape: {sample_batch[0].shape}")
print(f"Target shape: {sample_batch[1].shape}")
print(f"\nFirst sequence (input): {sample_batch[0][0][:10].tolist()}")
print(f"First sequence (target): {sample_batch[1][0][:10].tolist()}")


## Transformer Architecture Components

We'll use the same transformer architecture as for text generation:
1. Causal attention mask (prevent looking ahead)
2. Token and position embeddings
3. Multi-head attention
4. Feed-forward layers

In [ ]:
def causal_attention_mask(n_dest, n_src, device):
    """Create a causal mask to prevent attending to future positions"""
    i = torch.arange(n_dest, device=device).unsqueeze(1)
    j = torch.arange(n_src, device=device).unsqueeze(0)
    return i >= j

# Example usage:
mask = causal_attention_mask(10, 10, device)
print("Causal mask (position 2):", mask[2].tolist())


In [ ]:
class TokenAndPositionEmbedding(nn.Module):
    """Embed both the note tokens and their positions in the sequence"""
    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_len, embed_dim)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        pos_embeddings = self.pos_emb(positions)
        token_embeddings = self.token_emb(x)
        return token_embeddings + pos_embeddings


In [ ]:
class TransformerBlock(nn.Module):
    """Single transformer block with multi-head attention and feed-forward layers"""
    def __init__(self, num_heads, embed_dim, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout_rate, batch_first=True
        )
        self.ln_1 = nn.LayerNorm(embed_dim)
        self.dropout_1 = nn.Dropout(dropout_rate)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout_rate)
        )
        self.ln_2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        causal_mask = causal_attention_mask(seq_len, seq_len, x.device)

        # Multi-head attention with causal masking
        attn_output, attn_weights = self.attn(x, x, x, attn_mask=~causal_mask.bool())
        x = self.ln_1(x + self.dropout_1(attn_output))

        # Feed-forward network
        ffn_output = self.ffn(x)
        x = self.ln_2(x + ffn_output)

        return x, attn_weights


## Music Transformer Model

We now assemble all components into a complete music generation transformer.
This model will learn to predict the next note given a sequence of previous notes.

In [ ]:
class MusicTransformer(nn.Module):
    """Complete transformer model for music generation"""
    def __init__(self, max_len, vocab_size, embed_dim, num_heads, ff_dim, num_layers=2):
        super().__init__()
        self.embed = TokenAndPositionEmbedding(max_len, vocab_size, embed_dim)

        # Stack multiple transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(num_heads, embed_dim, ff_dim)
            for _ in range(num_layers)
        ])

        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.embed(x)

        attn_weights_list = []
        for transformer in self.transformer_blocks:
            x, attn_weights = transformer(x)
            attn_weights_list.append(attn_weights)

        logits = self.lm_head(x)
        return logits, attn_weights_list


## Training Function

We'll train the model using cross-entropy loss to predict the next note in each sequence.

In [ ]:
from tqdm import tqdm

def train_music_transformer(model, dataloader, optimizer, criterion, epochs, device):
    """Train the music transformer model"""
    model.to(device)
    model.train()

    loss_history = []

    for epoch in range(epochs):
        total_loss = 0

        data_loader_with_progress = tqdm(
            iterable=dataloader, ncols=120, desc=f"Epoch {epoch+1}/{epochs}"
        )

        for batch_number, (inputs, targets) in enumerate(data_loader_with_progress):
            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            logits, _ = model(inputs)

            # Compute loss
            loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if (batch_number % 50 == 0) or (batch_number == len(dataloader) - 1):
                avg_loss = total_loss / (batch_number + 1)
                data_loader_with_progress.set_postfix({"avg loss": f"{avg_loss:.4f}"})

        epoch_loss = total_loss / len(dataloader)
        loss_history.append(epoch_loss)
        print(f"Epoch {epoch+1}/{epochs} - Average Loss: {epoch_loss:.4f}")

    return loss_history


## Hyperparameters

Let's define the hyperparameters for our music transformer:

## Model Training

Now let's instantiate and train the model:

## Visualize Training Progress

Let's plot the training loss over epochs:

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(loss_history, marker='o')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()


## Music Generation

Now we can use the trained model to generate new musical sequences!
The generator will predict the next note based on the current sequence.

In [ ]:
class MusicGenerator:
    """Generate music sequences using the trained transformer"""
    def __init__(self, model, vocab_size, temperature=1.0):
        self.model = model
        self.model.to(device)
        self.vocab_size = vocab_size
        self.temperature = temperature

    def sample_from_logits(self, logits, temperature):
        """Sample next note from logits with temperature scaling"""
        # Apply temperature
        probs = torch.nn.functional.softmax(logits / temperature, dim=-1)

        # Sample from the distribution
        next_token = torch.multinomial(probs, num_samples=1).item()
        return next_token, probs

    def generate(self, seed_sequence, max_length=100, temperature=1.0):
        """
        Generate a music sequence

        Args:
            seed_sequence: Initial sequence of MIDI notes (list of ints)
            max_length: Maximum length of generated sequence
            temperature: Sampling temperature (higher = more random)
        """
        self.model.eval()
        generated = seed_sequence.copy()

        with torch.no_grad():
            for _ in range(max_length):
                # Prepare input (use last 32 notes as context)
                context = generated[-32:]
                x = torch.tensor([context], dtype=torch.long).to(device)

                # Get prediction
                logits, _ = self.model(x)
                last_logits = logits[0, -1]

                # Sample next note
                next_note, probs = self.sample_from_logits(last_logits, temperature)

                # Stop if we generate a pad token
                if next_note >= 128:
                    break

                generated.append(next_note)

        # Decode to note names
        generated_notes = [decode_note(n) for n in generated]

        print(f"Generated {len(generated)} notes")
        print(f"MIDI sequence: {generated}")

        return generated, generated_notes


## Generate Music

Let's generate some music! We'll start with a seed sequence (C major scale start)
and let the model continue the melody.

## Visualize Generated Music

Let's create a piano roll visualization of the generated sequences:

In [ ]:
def plot_piano_roll(sequence, title="Piano Roll"):
    """Plot a piano roll visualization of a MIDI sequence"""
    fig, ax = plt.subplots(figsize=(15, 4))

    # Filter out special tokens (>127)
    notes = [n for n in sequence if n < 128]

    # Create piano roll
    for i, note in enumerate(notes):
        if note > 0:  # Skip rests
            ax.plot([i, i+1], [note, note], 'b-', linewidth=8)

    ax.set_xlabel('Time Step')
    ax.set_ylabel('MIDI Note')
    ax.set_title(title)
    ax.set_ylim(50, 80)  # Focus on typical piano range
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

# Plot all three generations
fig1 = plot_piano_roll(generated_low, "Generated Music (Temperature=0.5)")
plt.show()

fig2 = plot_piano_roll(generated_med, "Generated Music (Temperature=1.0)")
plt.show()

fig3 = plot_piano_roll(generated_high, "Generated Music (Temperature=1.5)")
plt.show()


## Export to MIDI and Audio

Now let's export the generated music to MIDI files that can be played!

In [ ]:
import mido
from mido import Message, MidiFile, MidiTrack

def export_to_midi(notes, filename="generated_music.mid", tempo=500000, note_duration=480):
    mid = MidiFile()
    track = MidiTrack()
    mid.tracks.append(track)

    # Set tempo
    track.append(mido.MetaMessage('set_tempo', tempo=tempo))

    # Add notes
    for note in notes:
        if 0 < note < 128:  # Valid MIDI note
            track.append(Message('note_on', note=int(note), velocity=64, time=0))
            track.append(Message('note_off', note=int(note), velocity=64, time=note_duration))

    mid.save(filename)
    print(f"✓ MIDI file saved as '{filename}'")
    return filename
